# Radiomics experiments and results

Sections 1–5 explore training choices. The sections below calculate only the results used in [the main notebook](radiomics_analysis.ipynb). They read saved predictions and do not retrain models. Historical results include repeated CT copies. The newer binary ROC excludes the confirmed copies.

| Main section | Calculation | Output |
|---|---|---|
| 6.1 Five-class competition | [8.1](#81-five-class-table-for-the-main-notebook) | `radiomics_five_class_results.csv` |
| 6.1 Sphere vs segmentation | [8.2](#82-spheres-versus-segmentation-on-shared-nodules) | `radiomics_region_comparison.csv` |
| 6.2 Historical default decisions | [11.1](#111-default-decisions) | `radiomics_binary_default_results.csv` |
| 6.2 ROC | [11.8](#118-duplicate-excluded-roc-for-the-main-notebook) | `radiomics_binary_roc_points.csv`, `radiomics_binary_roc.png` |
| 6.3 Regression | [7.1](#71-regression-table-for-the-main-notebook) | `radiomics_regression_results.csv` |

The main notebook calculates its size and filtering comparisons directly. They are not repeated here.

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "configs/paths.yaml").exists(), "Run from the repository root or notebooks/."
sys.path.insert(0, str(ROOT))

from src.modeling import feature_inventory, EXCLUDED_LNDB_IDS, metrics

KEYS = ["LNDbID", "FindingID"]
LIDC_KEYS = ["patient_id", "series_uid", "nodule_id"]
CLASSES = [1, 2, 3, 4, 5]
print("Repository:", ROOT)
print("Duplicate CTs excluded only with --exclude-duplicate-lndb:", sorted(EXCLUDED_LNDB_IDS))

Repository: /home/jp/Documents/por-definir-ds
Duplicate CTs excluded only with --exclude-duplicate-lndb: [181, 182, 183, 184, 185, 188, 189, 194, 195, 198, 226]


## 1. What features do we have?

Training data only. Each row counts nodules with extracted features in that CSV.

In [3]:
from src.modeling import feature_inventory

#for 2.5D, each slice has its own row.
display(feature_inventory(ROOT / "data"))

,dataset,region,method,CSV,nodules,rows,features
0,LIDC,circle,2.5D,lidc_circle_r11_5slice_hu.csv,2637,13185,930
1,LIDC,circle,2D,lidc_circle_r11_bw10_hu.csv,2637,2637,930
2,LIDC,circle,2D,lidc_circle_r11_clipped.csv,2637,2637,930
3,LIDC,circle,2D,lidc_circle_r11_hu.csv,2637,2637,930
4,LIDC,circle,2D,lidc_circle_r14_clipped.csv,2637,2637,930
5,LIDC,circle,2D,lidc_circle_r14_hu.csv,2637,2637,930
6,LIDC,circle,2D,lidc_circle_r17_clipped.csv,2637,2637,930
7,LIDC,circle,2D,lidc_circle_r17_hu.csv,2637,2637,930
8,LIDC,circle,2D,lidc_circle_r21_clipped.csv,2637,2637,930
9,LIDC,circle,2D,lidc_circle_r21_hu.csv,2637,2637,930


## 2. What did we change between experiments?

### Circle radii

We start with circles extracted from the original HU values and validate on all 765 LNDb training nodules. Consensus applies to segmentation masks, not circles.

We used the P95, P97 and P99 nodule-size percentiles to choose circle radii for LNDb and LIDC.

| Percentile | LNDb radius | LIDC radius |
|---|---:|---:|
| P95 | 6 mm | 14 mm |
| P97 | 7 mm | 17 mm |
| P99 | 11 mm | 21 mm |

Because nodule sizes differ between the datasets, we tried using each dataset’s radius. We also tried using the same radius in both datasets, taking the LNDb value, to see whether keeping the extraction region the same size works better. We additionally tried 11 mm in LIDC with 6 mm in LNDb.

In [4]:
def display_models(models):
    scores = {}
    for choice, name in models.items():
        table = pd.read_csv(ROOT / "models" / name / "metrics.csv")
        scores[choice] = table.loc[
            table["fold"].eq("overall_oof"),
            ["balanced_accuracy", "macro_f1", "qwk", "mae"],
        ].iloc[0]

    display(pd.DataFrame(scores).T.round(3))

In [4]:
models = {
    "same radius: LNDb 6, LIDC 6": "rf_lidc_circle_r6_hu_lndb_circle_r6_hu_balanced",
    "same radius: LNDb 7, LIDC 7": "rf_lidc_circle_r7_hu_lndb_circle_r7_hu_balanced",
    "same radius: LNDb 11, LIDC 11": "rf_lidc_circle_r11_hu_lndb_circle_r11_hu_balanced",
    "P95: LNDb 6, LIDC 14": "rf_circle_r14_r6_balanced_5class",
    "P97: LNDb 7, LIDC 17": "rf_circle_r17_r7_balanced_5class",
    "P99: LNDb 11, LIDC 21": "rf_lidc_circle_r21_hu_lndb_circle_r11_hu_balanced",
    "extra: LNDb 6, LIDC 11": "rf_lidc_circle_r11_hu_lndb_circle_r6_hu_balanced",
}

display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"same radius: LNDb 6, LIDC 6",0.382,0.383,0.443,0.725
"same radius: LNDb 7, LIDC 7",0.388,0.385,0.445,0.724
"same radius: LNDb 11, LIDC 11",0.360,0.369,0.351,0.790
"P95: LNDb 6, LIDC 14",0.409,0.422,0.435,0.698
"P97: LNDb 7, LIDC 17",0.375,0.388,0.435,0.690
"P99: LNDb 11, LIDC 21",0.345,0.371,0.352,0.736
"extra: LNDb 6, LIDC 11",0.408,0.429,0.442,0.685


We thought P99 might help by including more of the perinodular region, but both P99 choices had lower balanced accuracy than P95. Larger circles may also include nearby vessels and other tissue. These scores cannot tell us why the predictions changed. The P95 14/6 mm model and the extra 11/6 mm model are nearly tied on balanced accuracy (0.409 and 0.408). The 11/6 mm model has lower MAE (0.685 versus 0.698).

### Does clipping help?

As described in the preprocessing notebook, we also extracted features after clipping CT values to [-1000, 400] HU. We compare this with the original HU values using the same 14/6 mm circles, model and 765 LNDb nodules.

In [5]:
clipping_models = {
    "14/6 mm, original HU": "rf_circle_r14_r6_balanced_5class",
    "14/6 mm, clipped HU": "rf_circle_r14_r6_clipped_balanced_5class",
}

display_models(clipping_models)

,balanced_accuracy,macro_f1,qwk,mae
"14/6 mm, original HU",0.409,0.422,0.435,0.698
"14/6 mm, clipped HU",0.356,0.363,0.421,0.707


Clipping did not help this 14/6 mm model. Balanced accuracy fell from 0.409 to 0.356, and MAE rose from 0.698 to 0.707. The original HU values also gave higher macro F1 and QWK here.

### Do circles or 2D segmentations work better?

The radius comparisons above used all 765 LNDb nodules. Segmentation masks are available only for nodules ≥3 mm, so this comparison uses 622 LNDb nodules. We retrained the 14/6 mm circle model on exactly the same LNDb and LIDC nodules as the 2D, 50% consensus segmentation model.

In [6]:
models = {
    "circle, 14/6 mm": "rf_circle_r14_r6_ge3_c50_population_balanced",
    "2D segmentation, 50% consensus": "rf_segmentation_2d_c50_bw25_available_balanced",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"circle, 14/6 mm",0.397,0.427,0.414,0.711
"2D segmentation, 50% consensus",0.399,0.414,0.529,0.650


Balanced accuracy is nearly the same (0.397 for circles and 0.399 for segmentation). Segmentation has higher QWK (0.529 versus 0.414) and lower MAE (0.650 versus 0.711), while circles have higher macro F1 (0.427 versus 0.414). Circles use the slice nearest the centroid, whereas segmentation uses the slice with the largest mask area, so the region and selected slice both change.

### What changes with 2D mask consensus?

A higher consensus requires more readers to include a pixel in the mask. With 75% and 100% consensus, some masks become too small to extract features. The 25% and 50% models use 622 LNDb nodules. The 75% and 100% models use 614, so their overall scores also reflect a change in the training and validation population.

In [7]:
models = {
    "25%": "rf_segmentation_2d_c25_bw25_available_balanced",
    "50%": "rf_segmentation_2d_c50_bw25_available_balanced",
    "75%": "rf_segmentation_2d_c75_bw25_available_balanced",
    "100%": "rf_segmentation_2d_c100_bw25_available_balanced",
}
display_models(models)

lndb_ids = []
for level in (75, 100):
    lidc = pd.read_csv(ROOT / "data" / f"lidc_segmentation_2d_c{level}_hu.csv")
    lndb = pd.read_csv(ROOT / "models" / models[f"{level}%"] / "validation_predictions.csv")
    lndb_ids.append(set(zip(lndb.LNDbID, lndb.FindingID)))
    print(f"{level}%: {len(lidc)} LIDC, {len(lndb)} LNDb")

print("Same LNDb nodules:", lndb_ids[0] == lndb_ids[1])

,balanced_accuracy,macro_f1,qwk,mae
25%,0.388,0.399,0.493,0.666
50%,0.399,0.414,0.529,0.650
75%,0.379,0.394,0.503,0.674
100%,0.375,0.388,0.484,0.686


75%: 2630 LIDC, 614 LNDb


100%: 2628 LIDC, 614 LNDb
Same LNDb nodules: True


The 50% consensus model has the highest balanced accuracy, macro F1 and QWK, and the lowest MAE in this table. The 75% and 100% models also lose eight LNDb nodules and some LIDC nodules during extraction. Their score differences cannot be attributed to the consensus level alone.

### Does 3D segmentation help?

Both models use 50% consensus masks and validate on the same 622 LNDb nodules. The 2D model extracts features from one axial slice, while the 3D model uses the full mask volume. The 3D extraction also includes one more LIDC training nodule.

In [8]:
models = {
    "2D": "rf_segmentation_2d_c50_bw25_available_balanced",
    "3D": "rf_segmentation_3d_c50_bw25_ge3_balanced",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
2D,0.399,0.414,0.529,0.650
3D,0.428,0.446,0.523,0.632


The 3D model has higher balanced accuracy (0.428 versus 0.399) and macro F1 (0.446 versus 0.414), and lower MAE (0.632 versus 0.650). The 2D model has slightly higher QWK (0.529 versus 0.523). Since the LIDC training populations differ by one nodule, this does not isolate the effect of 2D versus 3D extraction completely.

### Does a 3D sphere help?

At 11/6 mm, compare a circle on one slice with a 3D sphere around the same centroid. Both models use all 765 LNDb training nodules.

In [9]:
models = {
    "2D circle, 11/6 mm": "rf_lidc_circle_r11_hu_lndb_circle_r6_hu_balanced",
    "3D sphere, 11/6 mm": "rf_sphere_bw25_balanced",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"2D circle, 11/6 mm",0.408,0.429,0.442,0.685
"3D sphere, 11/6 mm",0.399,0.416,0.489,0.655


The sphere has higher QWK (0.489 versus 0.442) and lower MAE (0.655 versus 0.685), but lower balanced accuracy (0.399 versus 0.408).

### Do spheres or 3D segmentations work better?

To compare a sphere with the 3D, 50% consensus segmentation, restrict the sphere model to the same 622 LNDb nodules. The full-population sphere score above is not directly comparable with this segmentation score.

In [10]:
models = {
    "3D sphere, LNDb ≥3 mm": "rf_sphere_bw25_ge3_balanced",
    "3D segmentation, 50%": "rf_segmentation_3d_c50_bw25_ge3_balanced",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"3D sphere, LNDb ≥3 mm",0.401,0.428,0.475,0.680
"3D segmentation, 50%",0.428,0.446,0.523,0.632


On these 622 nodules, 3D segmentation improves balanced accuracy (0.428 versus 0.401), QWK (0.523 versus 0.475) and MAE (0.632 versus 0.680) over the sphere. Unlike the sphere, it uses the provided nodule mask.

### Do five circle slices help?

We also trained a five-class RF using five adjacent circle slices instead of one. Each training nodule contributes five rows, although validation still counts each nodule once.

In [11]:
models = {
    "one circle slice, 11/6 mm": "rf_lidc_circle_r11_hu_lndb_circle_r6_hu_balanced",
    "five circle slices, 11/6 mm": "rf_circle_r11_r6_5slice_balanced",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"one circle slice, 11/6 mm",0.408,0.429,0.442,0.685
"five circle slices, 11/6 mm",0.353,0.373,0.396,0.708


For five-class prediction, five slices reduce balanced accuracy from 0.408 to 0.353 and QWK from 0.442 to 0.396. The binary result is a separate question below.

### Does a smaller bin width help?

As described in the radiomics decisions notebook, bin width groups CT values for texture features. Here we compare 25 HU with 10 HU using the same 11/6 mm circles and 765 LNDb nodules.

In [12]:
models = {
    "25 HU": "rf_lidc_circle_r11_hu_lndb_circle_r6_hu_balanced",
    "10 HU": "rf_circle_r11_r6_bw10_balanced",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
25 HU,0.408,0.429,0.442,0.685
10 HU,0.370,0.382,0.437,0.695


For these 11/6 mm circles, 25 HU gives higher balanced accuracy (0.408 versus 0.370), macro F1 and QWK, and lower MAE (0.685 versus 0.695) than 10 HU. This comparison does not establish the best bin width for other regions or radii.

## 3. Does training on LIDC or changing class balance help?

We keep the LNDb circle radius at 6 mm and compare training on LNDb alone with training on LNDb plus LIDC, using 14 mm circles for LIDC. Both models are evaluated on the same 765 LNDb nodules.

In [13]:
models = {
    "LNDb only, 6 mm": "rf_lndb_circle_r6_hu_balanced",
    "LNDb + LIDC, 14/6 mm": "rf_circle_r14_r6_balanced_5class",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"LNDb only, 6 mm",0.409,0.430,0.446,0.681
"LNDb + LIDC, 14/6 mm",0.409,0.422,0.435,0.698


No big difference.


The five-class labels are uneven. Among the 765 LNDb nodules, 254 are class 2 but only 18 are class 5. We first compare RF with and without class weights, then class weights, SMOTE and Balanced RF on the same 14/6 mm circles.

For 3D spheres and 50% consensus segmentations, compare the three balancing methods separately on the same 622 LNDb nodules ≥3 mm. The sphere results here use the ≥3 mm model, not the 765-nodule sphere model.

In [14]:
models = {
    "no class weights": "rf_circle_r14_r6_unweighted_5class",
    "balanced class weights": "rf_circle_r14_r6_balanced_5class",
}
display_models(models)
models = {
    "LNDb only, no class weights": "rf_lndb_circle_r6_hu",
    "LNDb only, balanced": "rf_lndb_circle_r6_hu_balanced",
}
display_models(models)
models = {
    "class weights": "rf_circle_r14_r6_balanced_5class",
    "SMOTE": "rf_circle_r14_r6_smote_5class",
    "balanced RF": "brf_circle_r14_r6_5class",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
no class weights,0.385,0.402,0.467,0.667
balanced class weights,0.409,0.422,0.435,0.698


,balanced_accuracy,macro_f1,qwk,mae
"LNDb only, no class weights",0.379,0.393,0.458,0.672
"LNDb only, balanced",0.409,0.430,0.446,0.681


,balanced_accuracy,macro_f1,qwk,mae
class weights,0.409,0.422,0.435,0.698
SMOTE,0.412,0.399,0.439,0.736
balanced RF,0.426,0.323,0.414,0.809


In [15]:
models = {
    "class weights, 3D sphere": "rf_sphere_bw25_ge3_balanced",
    "SMOTE, 3D sphere": "rf_sphere_ge3_smote_5class",
    "Balanced RF, 3D sphere": "brf_sphere_ge3_5class",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"class weights, 3D sphere",0.401,0.428,0.475,0.680
"SMOTE, 3D sphere",0.452,0.436,0.500,0.722
"Balanced RF, 3D sphere",0.484,0.385,0.477,0.788


In [16]:
models = {
    "class weights, 3D segmentation": "rf_segmentation_3d_c50_bw25_ge3_balanced",
    "SMOTE, 3D segmentation": "rf_segmentation_3d_c50_ge3_smote_5class",
    "Balanced RF, 3D segmentation": "brf_segmentation_3d_c50_ge3_5class",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"class weights, 3D segmentation",0.428,0.446,0.523,0.632
"SMOTE, 3D segmentation",0.445,0.435,0.504,0.698
"Balanced RF, 3D segmentation",0.480,0.416,0.519,0.743


With 14/6 mm circles, Balanced RF has the highest balanced accuracy among these three balancing methods, but class-weighted RF has higher macro F1 and lower MAE. On both 3D regions, Balanced RF again has the highest balanced accuracy (0.484 for spheres and 0.480 for segmentations), while class-weighted RF has lower MAE (0.680 and 0.632). No balancing method wins on every metric.

## 4. Do fewer features help?

First compare all 14/6 mm circle features with top 100, top 500, Original-only and normalized pairs for each balancing method. Original-only uses features from the CT image without additional filters. Normalized pairs keeps other features, but removes a measure's non-normalized version when its `Normalized` version exists.

Then compare all features with top 100, 500 and 1000 for class-weighted RF on 11/6 mm 3D spheres and 50% consensus 3D segmentations. Balanced RF feature selection is also shown where trained. The 3D models use the same 622 LNDb nodules ≥3 mm, while circles use 765 nodules. Top-k uses [ANOVA F scores](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.f_classif.html) and is selected inside each training fold.

In [17]:
models = {
    "all, class weights": "rf_circle_r14_r6_balanced_5class",
    "top 100, class weights": "rf_circle_r14_r6_balanced_k100_5class",
    "top 500, class weights": "rf_circle_r14_r6_balanced_k500_5class",
    "Original only, class weights": "rf_circle_r14_r6_balanced_original_5class",
    "normalized pairs, class weights": "rf_circle_r14_r6_balanced_normalized_pairs_5class",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"all, class weights",0.409,0.422,0.435,0.698
"top 100, class weights",0.355,0.370,0.428,0.697
"top 500, class weights",0.396,0.416,0.432,0.699
"Original only, class weights",0.362,0.382,0.434,0.718
"normalized pairs, class weights",0.395,0.425,0.429,0.693


In [18]:
models = {
    "all, SMOTE": "rf_circle_r14_r6_smote_5class",
    "top 100, SMOTE": "rf_circle_r14_r6_smote_k100_5class",
    "top 500, SMOTE": "rf_circle_r14_r6_smote_k500_5class",
    "Original only, SMOTE": "rf_circle_r14_r6_smote_original_5class",
    "normalized pairs, SMOTE": "rf_circle_r14_r6_smote_normalized_pairs_5class",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"all, SMOTE",0.412,0.399,0.439,0.736
"top 100, SMOTE",0.340,0.323,0.424,0.766
"top 500, SMOTE",0.409,0.389,0.433,0.749
"Original only, SMOTE",0.359,0.329,0.403,0.841
"normalized pairs, SMOTE",0.405,0.396,0.430,0.750


In [19]:
models = {
    "all, Balanced RF": "brf_circle_r14_r6_5class",
    "top 100, Balanced RF": "brf_circle_r14_r6_k100_5class",
    "top 500, Balanced RF": "brf_circle_r14_r6_k500_5class",
    "Original only, Balanced RF": "brf_circle_r14_r6_original_5class",
    "normalized pairs, Balanced RF": "brf_circle_r14_r6_normalized_pairs_5class",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"all, Balanced RF",0.426,0.323,0.414,0.809
"top 100, Balanced RF",0.420,0.334,0.435,0.783
"top 500, Balanced RF",0.417,0.313,0.402,0.825
"Original only, Balanced RF",0.424,0.315,0.390,0.843
"normalized pairs, Balanced RF",0.434,0.332,0.419,0.799


In [20]:
models = {
    "all, 3D sphere": "rf_sphere_bw25_ge3_balanced",
    "top 100, 3D sphere": "rf_sphere_ge3_balanced_k100_5class",
    "top 500, 3D sphere": "rf_sphere_ge3_balanced_k500_5class",
    "top 1000, 3D sphere": "rf_sphere_ge3_balanced_k1000_5class",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"all, 3D sphere",0.401,0.428,0.475,0.680
"top 100, 3D sphere",0.414,0.443,0.485,0.670
"top 500, 3D sphere",0.410,0.432,0.492,0.677
"top 1000, 3D sphere",0.380,0.401,0.456,0.704


In [21]:
models = {
    "all, 3D sphere Balanced RF": "brf_sphere_ge3_5class",
    "top 100, 3D sphere Balanced RF": "brf_sphere_ge3_k100_5class",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"all, 3D sphere Balanced RF",0.484,0.385,0.477,0.788
"top 100, 3D sphere Balanced RF",0.466,0.387,0.482,0.778


In [22]:
models = {
    "all, 3D segmentation RF": "rf_segmentation_3d_c50_bw25_ge3_balanced",
    "top 100, 3D segmentation RF": "rf_segmentation_3d_c50_ge3_balanced_k100_5class",
    "top 500, 3D segmentation RF": "rf_segmentation_3d_c50_ge3_balanced_k500_5class",
    "top 1000, 3D segmentation RF": "rf_segmentation_3d_c50_ge3_balanced_k1000_5class",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"all, 3D segmentation RF",0.428,0.446,0.523,0.632
"top 100, 3D segmentation RF",0.425,0.447,0.504,0.638
"top 500, 3D segmentation RF",0.412,0.430,0.495,0.662
"top 1000, 3D segmentation RF",0.409,0.426,0.504,0.656


In [23]:
models = {
    "all, 3D segmentation Balanced RF": "brf_segmentation_3d_c50_ge3_5class",
    "top 100, 3D segmentation Balanced RF": "brf_segmentation_3d_c50_ge3_k100_5class",
    "top 500, 3D segmentation Balanced RF": "brf_segmentation_3d_c50_ge3_k500_5class",
    "top 1000, 3D segmentation Balanced RF": "brf_segmentation_3d_c50_ge3_k1000_5class",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"all, 3D segmentation Balanced RF",0.480,0.416,0.519,0.743
"top 100, 3D segmentation Balanced RF",0.456,0.415,0.520,0.719
"top 500, 3D segmentation Balanced RF",0.446,0.400,0.505,0.759
"top 1000, 3D segmentation Balanced RF",0.468,0.411,0.515,0.752


With circles, top-k selection does not improve balanced accuracy for class-weighted RF or SMOTE. Balanced RF improves from 0.426 to 0.434 with normalized pairs, but still has higher MAE than class-weighted RF.

For 3D spheres, class-weighted RF improves across all four metrics with top 100 (balanced accuracy 0.414 versus 0.401 with all features). For 3D segmentation, all features have the highest balanced accuracy within both RF settings (0.428 with class weights and 0.480 with Balanced RF). Feature reduction helps the sphere RF, but not the 3D segmentation models on balanced accuracy.

## 5. How do Balanced RF and XGBoost compare?

Compare both models using all features within each region: 14/6 mm circles on 765 LNDb nodules, then 11/6 mm 3D spheres and 50% consensus 3D segmentations on the same 622 LNDb nodules ≥3 mm. Balanced RF undersamples each tree, while XGBoost uses balanced sample weights, so this tests the complete model settings rather than the algorithm alone.

In [24]:
models = {
    "Balanced RF, 14/6 mm circles": "brf_circle_r14_r6_5class",
    "XGBoost, 14/6 mm circles": "xgboost_circle_r14_r6_balanced_5class",
}
display_models(models)

models = {
    "Balanced RF, 3D sphere": "brf_sphere_ge3_5class",
    "XGBoost, 3D sphere": "xgboost_sphere_ge3_balanced_5class",
}
display_models(models)

models = {
    "Balanced RF, 3D segmentation": "brf_segmentation_3d_c50_ge3_5class",
    "XGBoost, 3D segmentation": "xgboost_segmentation_3d_c50_ge3_balanced_5class",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"Balanced RF, 14/6 mm circles",0.426,0.323,0.414,0.809
"XGBoost, 14/6 mm circles",0.436,0.386,0.489,0.718


,balanced_accuracy,macro_f1,qwk,mae
"Balanced RF, 3D sphere",0.484,0.385,0.477,0.788
"XGBoost, 3D sphere",0.442,0.410,0.502,0.725


,balanced_accuracy,macro_f1,qwk,mae
"Balanced RF, 3D segmentation",0.480,0.416,0.519,0.743
"XGBoost, 3D segmentation",0.455,0.430,0.529,0.712


With circles, XGBoost outperforms Balanced RF on all four metrics. With 3D spheres and segmentations, Balanced RF has higher balanced accuracy (0.484 and 0.480 versus 0.442 and 0.455), while XGBoost has higher macro F1 and QWK and lower MAE. These are matched comparisons within each region, not across the 765- and 622-nodule populations.

### Earlier RF tuning at 11/6 mm

The saved comparison below tunes ordinary RF on 11/6 mm circles and also shows a separate normalized-pairs feature test. It does not tune the 14/6 mm or 3D Balanced RF and XGBoost models.

In [25]:
models = {
    "ordinary RF": "rf_lidc_circle_r11_hu_lndb_circle_r6_hu_balanced",
    "tuned RF": "rf_lidc_circle_r11_hu_lndb_circle_r6_hu_balanced_tuned",
    "normalized pairs": "rf_lidc_circle_r11_hu_lndb_circle_r6_hu_balanced_normalized_pairs",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
ordinary RF,0.408,0.429,0.442,0.685
tuned RF,0.406,0.414,0.447,0.686
normalized pairs,0.404,0.433,0.433,0.685


Tuning gives almost the same balanced accuracy as ordinary RF (0.406 versus 0.408), with a small QWK increase (0.447 versus 0.442). Normalized pairs give slightly higher macro F1 (0.433 versus 0.429) but lower balanced accuracy and QWK. MAE is 0.685 for both ordinary RF and normalized pairs.

## 6. Median vs Mean in regression

In [5]:
display_models({
    "Ridge trained on mean": "ridge_segmentation_3d_c50_ge3",
    "Ridge trained on median": "ridge_segmentation_3d_c50_ge3_median",
})

,balanced_accuracy,macro_f1,qwk,mae
Ridge trained on mean,0.323,0.342,0.427,0.733
Ridge trained on median,0.309,0.322,0.413,0.743


## 7. Does regression provide useful alerts?

Ridge predicts the readers' mean rating. The three saved runs below use the same 622 nodules and patient folds. Continuous error compares the unrounded predictions with the mean rating. QWK compares predictions rounded to ratings 1–5.

For binary alerts, restrict the segmented Ridge model to the same historical 471 nodules used by the default XGBoost model. An alert is raised when Ridge predicts a mean above 3. The comparisons are between reader ratings, not confirmed diagnoses.

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score
from src.binary_evaluation import binary_metrics
from src.regression import regression_metrics

ridge_runs = {
    "Circle": "ridge_circle_r14_r6_ge3",
    "Sphere": "ridge_sphere_ge3",
    "Segmentation": "ridge_segmentation_3d_c50_ge3",
}
regression_rows = []
ridge_reference = None
for region, run in ridge_runs.items():
    rows = pd.read_csv(ROOT / "models" / run / "validation_predictions.csv").set_index(KEYS).sort_index()
    cohort = rows[["fold", "target_classification_mean", "target_regression_mean"]]
    assert len(rows) == 622 and rows.index.is_unique
    if ridge_reference is None:
        ridge_reference = cohort
    pd.testing.assert_frame_equal(ridge_reference, cohort, check_dtype=False)
    error = regression_metrics(rows.target_regression_mean, rows.prediction_continuous)
    regression_rows.append({
        "region": region, "run": run, "nodules": len(rows),
        "continuous MAE": error["mae"], "continuous RMSE": error["rmse"],
        "rounded five-class QWK": metrics(rows.target_classification_mean, rows.prediction)["qwk"],
    })

ridge = pd.read_csv(ROOT / "models" / ridge_runs["Segmentation"] / "validation_predictions.csv").set_index(KEYS).sort_index()
forest_run = "rf_segmentation_3d_c50_bw25_ge3_balanced"
forest = pd.read_csv(ROOT / "models" / forest_run / "validation_predictions.csv").set_index(KEYS).sort_index()
pd.testing.assert_frame_equal(ridge[["fold", "target_classification_mean"]],
                              forest[["fold", "target_classification_mean"]], check_dtype=False)
five_class_rows = []
for name, run, predicted in [("Ridge, rounded", ridge_runs["Segmentation"], ridge.prediction),
                             ("Class-weighted RF", forest_run, forest.prediction)]:
    result = metrics(ridge.target_classification_mean, predicted)
    five_class_rows.append({"model": name, "run": run, "nodules": len(ridge),
                            "balanced accuracy": result["balanced_accuracy"],
                            "macro F1": result["macro_f1"], "QWK": result["qwk"], "MAE": result["mae"]})

binary_run = "xgboost_segmentation_3d_c50_ge3_balanced_binary_roc"
xgboost = pd.read_csv(ROOT / "models" / binary_run / "validation_predictions.csv").set_index(KEYS).sort_index()
shared = ridge.loc[xgboost.index]
assert len(shared) == 471 and xgboost.index.is_unique
pd.testing.assert_frame_equal(shared[["fold"]], xgboost[["fold"]], check_dtype=False)
assert shared.target_regression_mean.gt(3).astype(int).equals(xgboost.target_binary_mean.astype(int))
binary_rows = []
for name, run, predicted, score in [
    ("Ridge, prediction >3", ridge_runs["Segmentation"], shared.prediction_continuous.gt(3).astype(int), shared.prediction_continuous),
    ("Binary XGBoost", binary_run, xgboost.prediction, xgboost.proba_1),
]:
    result = binary_metrics(xgboost.target_binary_mean, predicted)
    binary_rows.append({"model": name, "run": run, "nodules": len(shared),
                        "FN": result["FN"], "FP": result["FP"], "accuracy": result["accuracy"],
                        "balanced accuracy": result["balanced_accuracy"],
                        "macro F1": f1_score(xgboost.target_binary_mean, predicted, average="macro"),
                        "recall": result["sensitivity"], "AUC": roc_auc_score(xgboost.target_binary_mean, score)})

high = shared.target_classification_mean.ge(4)
assert int((high & shared.prediction.lt(4)).sum()) == 60
assert int((high & shared.prediction_continuous.le(3)).sum()) == 38
display(pd.DataFrame(five_class_rows).set_index("model").drop(columns=["run", "nodules"]).round(3))
display(pd.DataFrame(binary_rows).set_index("model").drop(columns=["run", "nodules"]).round(3))

### 7.1 Regression table for the main notebook

The first CSV gives continuous error and rounded QWK for the three Ridge regions, on 622 shared nodules. The two comparison CSVs put segmented Ridge beside the classification models on shared nodules. The five-class comparison uses 622 nodules. The binary comparison uses 471 historical nodules.

Source: `models/<run>/validation_predictions.csv`, with each source run named in its output CSV.

In [ ]:
reports = {
    "radiomics_regression_results.csv": pd.DataFrame(regression_rows),
    "radiomics_regression_five_class.csv": pd.DataFrame(five_class_rows),
    "radiomics_regression_binary.csv": pd.DataFrame(binary_rows),
}
for name, table in reports.items():
    table.to_csv(ROOT / "data/processed" / name, index=False)
    print("Saved:", name)

## 8. Five-class results

The 765-nodule competition ranking and the 622-nodule region comparison are separate populations. Do not compare their scores as though they came from the same nodules.

### 8.1 Five-class table for the main notebook

Rank saved five-class runs evaluated on the same 765 LNDb nodules and patient folds. The output lists the source run for each row. The main notebook shows the first five rows.

Source: `models/<run>/validation_predictions.csv` and `data/processed/lndb_nodule_targets.csv`.

In [ ]:
def add_combined_score(table):
    table = table.copy()
    table["combined score"] = (
        table["balanced_accuracy"] + table["macro_f1"]
        + table["qwk"] + (1 - table["mae"])
    ) / 4
    return table

targets = pd.read_csv(ROOT / "data/processed/lndb_nodule_targets.csv").rename(columns={"NoduleID": "FindingID"})
folds = pd.read_csv(ROOT / "data/processed/lndb_patient_folds.csv")
reference = targets.merge(folds, on="LNDbID", validate="many_to_one").set_index(KEYS).sort_index()
assert len(reference) == 765 and reference.index.is_unique

ranking_rows = []
for file in sorted((ROOT / "models").glob("*/validation_predictions.csv")):
    if "grid4" in file.parent.name and not (file.parent / "tuning.csv").exists():
        continue
    predictions = pd.read_csv(file)
    if len(predictions) != 765 or "target_classification_mean" not in predictions:
        continue
    assert not predictions.duplicated(KEYS).any(), file.parent.name
    predictions = predictions.set_index(KEYS).sort_index()
    if not predictions[["fold", "target_classification_mean"]].equals(
        reference[["fold", "target_classification_mean"]]
    ):
        continue
    assert predictions.prediction.isin(CLASSES).all(), file.parent.name
    ranking_rows.append({"run": file.parent.name,
                         **metrics(reference.target_classification_mean, predictions.prediction)})

ranking = add_combined_score(pd.DataFrame(ranking_rows)).sort_values("combined score", ascending=False)
names = {
    "rf_sphere_smote_5class": "RF + SMOTE, sphere",
    "rf_sphere_smote_aux": "RF + SMOTE + auxiliary targets, sphere",
    "rf_sphere_balanced_k100_5class": "Class-weighted RF, sphere, top 100",
    "expert_sphere_gldm": "RF + SMOTE, sphere, GLDM features",
    "expert_sphere_firstorder": "RF + SMOTE, sphere, intensity features",
}
ranking.insert(0, "model", ranking.run.map(names).fillna(ranking.run))
ranking["nodules"] = 765
ranking = ranking.rename(columns={"balanced_accuracy": "balanced accuracy", "macro_f1": "macro F1",
                                  "qwk": "QWK", "mae": "MAE"})
ranking.to_csv(ROOT / "data/processed/radiomics_five_class_results.csv", index=False)
display(ranking.set_index("model").drop(columns=["run", "nodules"]).head(5).round(3))

### 8.2 Spheres versus segmentation on shared nodules

Compare spheres and 3D segmentation in four matching training settings. All eight saved runs use the same 622 LNDb nodules, targets and patient folds.

Source: `models/<run>/validation_predictions.csv`. Output: `data/processed/radiomics_region_comparison.csv`.

In [ ]:
region_runs = {
    ("Class-weighted RF", "Sphere"): "rf_sphere_bw25_ge3_balanced",
    ("Class-weighted RF", "Segmentation"): "rf_segmentation_3d_c50_bw25_ge3_balanced",
    ("RF + SMOTE", "Sphere"): "rf_sphere_ge3_smote_5class",
    ("RF + SMOTE", "Segmentation"): "rf_segmentation_3d_c50_ge3_smote_5class",
    ("Balanced RF", "Sphere"): "brf_sphere_ge3_5class",
    ("Balanced RF", "Segmentation"): "brf_segmentation_3d_c50_ge3_5class",
    ("Class-weighted XGBoost", "Sphere"): "xgboost_sphere_ge3_balanced_5class",
    ("Class-weighted XGBoost", "Segmentation"): "xgboost_segmentation_3d_c50_ge3_balanced_5class",
}
region_rows = []
region_reference = None
for (model, region), run in region_runs.items():
    rows = pd.read_csv(ROOT / "models" / run / "validation_predictions.csv").set_index(KEYS).sort_index()
    cohort = rows[["fold", "target_classification_mean"]]
    assert len(rows) == 622 and rows.index.is_unique
    if region_reference is None:
        region_reference = cohort
    pd.testing.assert_frame_equal(region_reference, cohort, check_dtype=False)
    result = metrics(rows.target_classification_mean, rows.prediction)
    region_rows.append({"model": model, "region": region, "run": run, "nodules": len(rows), **result})

region_report = add_combined_score(pd.DataFrame(region_rows)).rename(columns={
    "balanced_accuracy": "balanced accuracy", "macro_f1": "macro F1", "qwk": "QWK", "mae": "MAE",
})
region_report.to_csv(ROOT / "data/processed/radiomics_region_comparison.csv", index=False)
display(region_report.set_index(["model", "region"]).drop(columns=["run", "nodules"]).round(3))

## 11. Binary results

The historical default decisions below include confirmed CT copies. The newer ROC excludes them. Both use only 3D segmentation features and reader mean >3 as the positive label.

### 11.1 Default decisions

Three saved binary runs on the same 471 LNDb nodules, including 91 positives. These historical runs include repeated CT copies. The default model decisions are used, not thresholds chosen to miss zero positives.

Source: `models/<run>/validation_predictions.csv`. Output: `data/processed/radiomics_binary_default_results.csv`.

In [ ]:
default_runs = {
    "XGBoost": "xgboost_segmentation_3d_c50_ge3_balanced_binary_roc",
    "Balanced RF": "brf_segmentation_3d_c50_ge3_binary_roc",
    "RF, intensity only": "rf_segmentation_firstorder_balanced_binary",
}
default_rows = []
default_reference = None
for name, run in default_runs.items():
    rows = pd.read_csv(ROOT / "models" / run / "validation_predictions.csv").set_index(KEYS).sort_index()
    cohort = rows[["fold", "target_binary_mean"]]
    assert len(rows) == 471 and rows.index.is_unique and rows.target_binary_mean.sum() == 91
    if default_reference is None:
        default_reference = cohort
    pd.testing.assert_frame_equal(default_reference, cohort, check_dtype=False)
    result = binary_metrics(rows.target_binary_mean, rows.prediction)
    default_rows.append({"model": name, "FN": result["FN"], "FP": result["FP"],
                         "accuracy": result["accuracy"], "balanced accuracy": result["balanced_accuracy"],
                         "macro F1": f1_score(rows.target_binary_mean, rows.prediction, average="macro"),
                         "recall": result["sensitivity"], "specificity": result["specificity"],
                         "precision": result["precision"], "AUC": roc_auc_score(rows.target_binary_mean, rows.proba_1),
                         "run": run, "nodules": len(rows), "positives": int(rows.target_binary_mean.sum())})

default_report = pd.DataFrame(default_rows)
default_report.to_csv(ROOT / "data/processed/radiomics_binary_default_results.csv", index=False)
display(default_report.set_index("model").drop(columns=["run", "nodules", "positives"]).round(3))

### 11.8 Duplicate-excluded ROC for the main notebook

Three baseline curves and two curves with the 0.7 diameter-correlation filter and tuning use the same 459 nodules, including 85 positives. Only 3D segmentation models are shown. The saved scores come from five-fold evaluation after confirmed CT copies were excluded.

Each marked threshold is chosen from the evaluated scores so no positive is missed **on that curve**. This is not a threshold validated on new nodules. At those marked points, baseline XGBoost has the fewest false alarms.

Source: `models/segmentation_binary_calibration/validation_predictions.csv` and `models/segmentation_binary_diameter07_tuned/validation_predictions.csv`. Outputs: `data/processed/radiomics_binary_roc_points.csv` and `data/processed/radiomics_binary_roc.png`.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
from src.binary_evaluation import zero_fn_summary
from src.modeling import EXCLUDED_LNDB_IDS

roc_runs = {
    "Baseline": "segmentation_binary_calibration",
    "Filtered + tuned": "segmentation_binary_diameter07_tuned",
}
roc_names = {
    "Binary XGBoost, all features": "XGBoost",
    "Binary Balanced RF, all features": "Balanced RF",
    "Binary RF, intensity": "RF, intensity only",
}
roc_targets = pd.read_csv(ROOT / "data/processed/lndb_ge3_nodule_targets.csv").rename(columns={"NoduleID": "FindingID"})
roc_folds = pd.read_csv(ROOT / "data/processed/lndb_patient_folds.csv")
roc_reference = roc_targets.merge(roc_folds, on="LNDbID", validate="many_to_one")
roc_reference = roc_reference.loc[roc_reference.target_binary_mean.notna()
                                  & ~roc_reference.LNDbID.isin(EXCLUDED_LNDB_IDS)]
roc_reference = roc_reference.set_index(KEYS).sort_index()[["fold", "target_binary_mean"]]
assert len(roc_reference) == 459 and roc_reference.target_binary_mean.sum() == 85

roc_rows = []
fig, ax = plt.subplots(figsize=(8, 5))
for variant, run in roc_runs.items():
    saved = pd.read_csv(ROOT / "models" / run / "validation_predictions.csv")
    for recipe, model in roc_names.items():
        if variant != "Baseline" and model == "RF, intensity only":
            continue
        group = saved.loc[saved.model.eq(recipe)].set_index(KEYS).sort_index()
        assert group.index.is_unique
        pd.testing.assert_frame_equal(roc_reference, group[["fold", "target_binary_mean"]], check_dtype=False)
        point = zero_fn_summary(group.target_binary_mean, group.score)
        predicted = group.score.ge(point["threshold"])
        roc_rows.append({"model": model, "variant": variant, "run": run, "recipe": recipe,
                         "positives": 85, "threshold selection": "evaluated rows", **point,
                         "macro F1": f1_score(group.target_binary_mean, predicted, average="macro")})
        false_positive_rate, recall, _ = roc_curve(group.target_binary_mean, group.score, drop_intermediate=False)
        line, = ax.plot(false_positive_rate, recall, linestyle="-" if variant == "Baseline" else "--",
                        label=f"{model}, {variant} (AUC {point['AUC']:.3f})")
        ax.scatter(1 - point["specificity"], 1, color=line.get_color(), s=35)

ax.plot([0, 1], [0, 1], linestyle=":", color="gray", linewidth=1)
ax.set(xlabel="False-positive rate (1 − specificity)", ylabel="Recall",
       title="Binary ROC: 459 nodules, confirmed CT copies excluded", xlim=(0, 1), ylim=(0, 1.03))
ax.legend(fontsize=8, loc="lower right")
ax.grid(alpha=0.2)
fig.tight_layout()
figure_path = ROOT / "data/processed/radiomics_binary_roc.png"
fig.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.show()

roc_report = pd.DataFrame(roc_rows).rename(columns={"balanced_accuracy": "balanced accuracy",
                                                    "sensitivity": "recall"})
roc_report.to_csv(ROOT / "data/processed/radiomics_binary_roc_points.csv", index=False)
display(roc_report.set_index(["model", "variant"])[[
    "threshold", "FN", "FP", "accuracy", "specificity",
]].round(4))